In [1]:
import argparse
import time
import numpy as np
import wandb

from reasoning.tools.utils import load_model_with_vllm
from datasets import load_dataset
from vllm import SamplingParams
from reasoning.inference.majority import MajorityInference
from reasoning.tools.logger import load_config, apply_config
import numpy as np
import argparse

from reasoning.evaluator.math_grader import math_equal, extract_answer

from reasoning.models.model import ValueModel_qwen


In [2]:


# Does not support batch inference



config = "/ssdscratch/byuan48/efficient_reasoning/configs/samplingTree.yaml"
config = load_config(config)
apply_config(config) 
print(config)

# load dataset and model
dataset = load_dataset(config["data_path"])
dataset = dataset['test']
model_name = config["model_name"]

policy_model, tokenizer = load_model_with_vllm(model_name, task='auto', tensor_parallel_size=len(config["cuda_device_ids"]), gpu_memory_utilization=0.8)
tokenizer.pad_token = tokenizer.eos_token 
reward_model = config.get("reward_model", None)
reward_model = ValueModel_qwen(device = "auto")

# inference hyperparameters
num_return_sequences = config["num_return_sequences"]
assert num_return_sequences == 1
max_new_tokens = config["max_new_tokens"]
temperature = config["temperature"]
top_p = config["top_p"]
system_prompt = config["system_prompt"]

# SamplingTree parameters
sampling_method = config["sampling_method"]
samplingTree_temperature = config["samplingTree_temperature"]
beam_width = config["beam_width"]
max_steps = config["max_steps"]

# name of the results file
config_name = config["config_name"]

# Also set the seed for the sampling params 
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_new_tokens,
    n=num_return_sequences,
    top_p=top_p,
    stop_token_ids=[tokenizer.eos_token_id],
    skip_special_tokens = True,
    include_stop_str_in_output = False,
) # shouldn't set seed for random sampling


wandb: Currently logged in as: yuanbo096. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'config_name': 'Llama-3.2-3B-Instruct-samplingTree_16', 'batch_size': 1, 'cuda_device_ids': [0, 1, 2, 3], 'seed': 42, 'data_path': 'HuggingFaceH4/MATH-500', 'model_name': 'meta-llama/Llama-3.2-3B-Instruct', 'system_prompt': 'Solve the following reasoning problem, strictly following the instructions below:\n1. Show your reasoning step by step. Each step should start with "## Step X:"\n2. Put your final answer within \\boxed{} at the end\n', 'max_new_tokens': 2048, 'temperature': 0.7, 'top_p': 0.9, 'experiment_name': 'samplingTree_experiment', 'inference_method': 'samplingTree', 'num_return_sequences': 1, 'wandb_project': 'efficient_reasoning', 'sampling_method': 'stochastic_beam_search', 'samplingTree_temperature': 0.1, 'beam_width': 4, 'max_steps': 5}
INFO 04-13 16:43:38 config.py:510] This model supports multiple tasks: {'reward', 'generate', 'score', 'embed', 'classify'}. Defaulting to 'generate'.
INFO 04-13 16:43:38 config.py:1310] Defaulting to use mp for distributed inference
WAR

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-13 16:43:43 model_runner.py:1099] Loading model weights took 1.5345 GB
(VllmWorkerProcess pid=3970984) INFO 04-13 16:43:43 model_runner.py:1099] Loading model weights took 1.5345 GB
(VllmWorkerProcess pid=3970986) INFO 04-13 16:43:43 model_runner.py:1099] Loading model weights took 1.5345 GB
(VllmWorkerProcess pid=3970985) INFO 04-13 16:43:44 model_runner.py:1099] Loading model weights took 1.5345 GB
(VllmWorkerProcess pid=3970985) (VllmWorkerProcess pid=3970986) (VllmWorkerProcess pid=3970984) INFO 04-13 16:43:47 worker.py:241] Memory profiling takes 3.18 seconds
INFO 04-13 16:43:47 worker.py:241] Memory profiling takes 3.18 seconds
(VllmWorkerProcess pid=3970986) INFO 04-13 16:43:47 worker.py:241] Memory profiling takes 3.18 seconds
(VllmWorkerProcess pid=3970985) INFO 04-13 16:43:47 worker.py:241] the current vLLM instance can use total_gpu_memory (47.43GiB) x gpu_memory_utilization (0.80) = 37.95GiB
(VllmWorkerProcess pid=3970984) INFO 04-13 16:43:47 worker.py:241] the curr

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 04-13 16:43:53 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
(VllmWorkerProcess pid=3970984) (VllmWorkerProcess pid=3970985) INFO 04-13 16:43:53 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 04-13 16:43:53 model_runner.py:1415] Captur

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:19<00:00,  1.77it/s]

(VllmWorkerProcess pid=3970984) INFO 04-13 16:44:13 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.40 GiB
INFO 04-13 16:44:13 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.40 GiB
(VllmWorkerProcess pid=3970985) INFO 04-13 16:44:13 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.40 GiB
(VllmWorkerProcess pid=3970986) INFO 04-13 16:44:13 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.40 GiB
INFO 04-13 16:44:13 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 29.67 seconds


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at Qwen/Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
import importlib
import reasoning.inference.tree
tree_module = importlib.reload(reasoning.inference.tree)
Tree = tree_module.Tree

In [4]:

start_time = time.time()
right_count = 0
num_generated_tokens = 0
for i in range(0, len(dataset)):
    question = dataset['problem'][i]
    answer = dataset['solution'][i]        

    tree = Tree(system_prompt, question, policy_model, reward_model, sampling_method, config_name, sampling_params, samplingTree_temperature, beam_width)
    for _ in range(max_steps):
        path = tree.generate_next_trajectory() 
        tree.paths.append(path)
        print(len(tree.paths))
    majority_inference = MajorityInference(policy_model=policy_model, tokenizer=tokenizer, sampling_params=sampling_params, config_name=config_name, reward_model=reward_model, method = 'weighted_majority')
    all_solutions  = [path.solutions for path in tree.explored_paths]
    rewards = [np.min(path.scores) for path in tree.explored_paths]
    
    solution = majority_inference.weighted_majority_vote(all_solutions, rewards)   
    extracted_answer = extract_answer(answer)
    is_correct = math_equal(extracted_answer, solution)
    if extracted_answer is None:
        raise ValueError('extracted answer is None')
    if is_correct:
        right_count += 1
    num_generated_tokens += tree.num_generated_tokens
    print("--------------------------------")
    print(f'question: {question}')
    print(f'solution: {solution}')
    print(f'extracted answer: {extracted_answer}')
    print(f'is correct: {is_correct}')
    print(f'Accuracy: {right_count/(i+1)}')
    print(f'Number of generated tokens: {tree.num_generated_tokens}')
    print("--------------------------------")


wandb.log({"Accuracy": right_count/len(dataset)})
wandb.log({"Number of generated tokens": num_generated_tokens/len(dataset)})
end_time = time.time()
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it, est. speed input: 86.46 toks/s, output: 166.16 toks/s]


1


Processed prompts: 100%|██████████| 4/4 [00:02<00:00,  1.74it/s, est. speed input: 402.65 toks/s, output: 298.83 toks/s] 


best_path: ['To convert the given point from rectangular coordinates to polar coordinates, we need to find the radius (r) and the angle (\\theta) that correspond to the point (0,3).\nWe can use the formulas $r = \\sqrt{x^2 + y^2}$ to find the radius and $\\theta = \\tan^{-1}(\\frac{y}{x})$ to find the angle. Calculate the radius (r) using the given rectangular coordinates (0,3).\n$r = \\sqrt{0^2 + 3^2} = \\sqrt{0 + 9} = \\sqrt{9} = 3$ Calculate the angle (\\theta) using the given rectangular coordinates (0,3).\nSince the point is on the y-axis, the angle is $\\theta = \\frac{\\pi}{2}$.\n \nTherefore, the polar coordinates are $\\left(3, \\frac{\\pi}{2}\\right)$.\n\n\\boxed{\\left(3, \\frac{\\pi}{2}\\right)}']
2


Processed prompts: 100%|██████████| 4/4 [00:02<00:00,  1.57it/s, est. speed input: 199.94 toks/s, output: 593.90 toks/s]


...

Processed prompts: 100%|██████████| 1/1 [00:11<00:00, 11.92s/it, est. speed input: 21.23 toks/s, output: 171.85 toks/s]


1


Processed prompts: 100%|██████████| 4/4 [00:13<00:00,  3.34s/it, est. speed input: 324.67 toks/s, output: 503.75 toks/s]


best_path: ['Expand the given equations to express the differences between the variables in terms of squares\nThe given equations can be expanded to express the differences between the variables $x_i$ and $y_i$ in terms of squares as follows:\n\\begin{align*}\n(x_1 - x_2)^2 + (y_1 - y_2)^2 &= 9 \\\\\nx_1^2 - 2x_1x_2 + x_2^2 + y_1^2 - 2y_1y_2 + y_2^2 &= 9 \\\\\nx_1^2 + y_1^2 - 2x_1x_2 - 2y_1y_2 + x_2^2 + y_2^2 &= 9 \\\\\nx_1^2 + y_1^2 + x_2^2 + y_2^2 - 2x_1x_2 - 2y_1y_2 &= 9 \\tag{1} \\label{eq1}\n\\end{align*}\n\n\\begin{align*}\n(x_1 - x_3)^2 + (y_1 - y_3)^2 &= 16 \\\\\nx_1^2 - 2x_1x_3 + x_3^2 + y_1^2 - 2y_1y_3 + y_3^2 &= 16 \\\\\nx_1^2 + y_1^2 - 2x_1x_3 - 2y_1y_3 + x_3^2 + y_3^2 &= 16 \\\\\nx_1^2 + y_1^2 + x_3^2 + y_3^2 - 2x_1x_3 - 2y_1y_3 &= 16 \\tag{2} \\label{eq2}\n\\end{align*}\n\n\\begin{align*}\n(x_2 - x_3)^2 + (y_2 - y_3)^2 &= 25 \\\\\nx_2^2 - 2x_2x_3 + x_3^2 + y_2^2 - 2y_2y_3 + y_3^2 &= 25 \\\\\nx_2^2 + y_2^2 - 2x_2x_3 - 2y_2y_3 + x_3^2 + y_3^2 &= 25 \\\\\nx_2^2 + y_2^2 + x_3

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]